In [4]:
import sys
sys.path.append('/app')

from src.climate_learn import convert_nc2npz, IterDataModule
from src.climate_learn.utils import load_downscaling_module
import numpy as np
import os
import glob
import xarray as xr
import xesmf

from IPython.display import HTML
import pytorch_lightning as pl
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger

## Data processing

We have slightly modified the original ClimateLearn procedure to preprocess raw data, but it remains largely unchanged. To execute it, choose the subsection below corresponding to the desired task.

In [ ]:
# Function to coarsen the data with scale factor, aka upsampling (High resolution --> Low resolution)
def coarsen_data(root_dir, target_dir, SCALE):
    folders = os.listdir(root_dir)

    # Random file to define coarsening regrid
    file_HR = os.listdir(os.path.join(root_dir, folders[0]))[0]

    engine="zarr" if "zarr" in file_HR else "netcdf4"
    ds_in = xr.open_dataset(os.path.join(root_dir, folders[0], file_HR), engine=engine)

    lat_axis = [k for k in list(ds_in.dims) if 'lat' in k][0]
    lon_axis = [k for k in list(ds_in.dims) if 'lon' in k][0]

    lon_new = ds_in[lon_axis].values[::SCALE]
    lat_new = ds_in[lat_axis].values[::SCALE]

    if "cmip" in root_dir:
        ds_out = xr.Dataset(
                coords=dict( 
                    lon=([lon_axis], lon_new),
                    lat=([lat_axis], lat_new),
                    time = ds_in.time.values)
            )
        periodic=True
    elif "obs" in root_dir:
        ds_out = xr.Dataset(
                coords=dict( 
                    longitude=([lon_axis], lon_new),
                    latitude=([lat_axis], lat_new),
                    time = ds_in.time.values)
            )
        periodic=False
    print(periodic)
    regridder = xesmf.Regridder(ds_in, ds_out, 'bilinear', periodic=periodic)
    folders = [f for f in folders if ".nc" not in f]
    # Write regridded file
    for f in folders:
        file_list = os.listdir(os.path.join(root_dir, f))
        os.makedirs(os.path.join(target_dir, f), exist_ok=True)
        
        for file in file_list:

            engine="zarr" if "zarr" in file else "netcdf4"
            ds_in =  xr.open_dataset(os.path.join(root_dir, f, file), engine=engine)
            try:
                ds_out = regridder(ds_in, keep_attrs=True)
            except ValueError:
                # Exception caused by E-OBS data resolution inconsistency
                print("Got regrid exception")
                regridder = xesmf.Regridder(ds_in, ds_out, 'bilinear', periodic=periodic)
                ds_out = regridder(ds_in, keep_attrs=True)
            
            if engine=="zarr":
                file_name = file.split("_")[-1]
                ds_out.to_zarr(os.path.join(target_dir, f, file_name), mode = 'w')
            elif "cmip" in root_dir:
                file_name = file.split("_")[-1]
                ds_out.to_netcdf(os.path.join(target_dir, f, file_name), mode = 'w')
            elif "obs" in root_dir:
                file_name = file.split("_")[0]+".nc"
                ds_out.to_netcdf(os.path.join(target_dir, f, file_name), mode = 'w')
            print(f, file_name)
        

### CMIP_CMIP task

In [ ]:
# Create LR images with simple bilinear upscaling
coarsen_data(root_dir="/app/data/raw/cmip6-cmip6/HR",
        target_dir = "/app/data/raw/cmip6-cmip6/LR_factor2",
        SCALE = 2)

In [ ]:
# Process data for CMIP_CMIP task
convert_nc2npz(
    root_dir="/app/data/raw/cmip6-cmip6/HR",
    save_dir="/app/data/processed/cmip6-cmip6/HR",
    src="cmip6",
    variables=[
                "air_temperature",
                "u_component_of_wind",
                "v_component_of_wind",
                "precipitation",
                "pressure_sea_level", # "surface_pressure" for 3H | "pressure_sea_level" for "D"
                "specific_humidity",
                "cloud_cover", 
                "upward_heat_flux",
                "moisture_in_soil"
               ],
    start_train_year=1960,
    start_val_year=2011,
    start_test_year=2013,
    end_year=2015,
    num_shards=5, # set 5 for "D" and 20 for "3H"
    frequency="D", # H | 3H | D
    align_target = None
)

100%|██████████| 2/2 [02:59<00:00, 89.88s/it]


In [ ]:
# CMIP data for CMIP_CMIP task
convert_nc2npz(
    root_dir="/app/data/raw/cmip6-cmip6/LR_factor2",
    save_dir="/app/data/processed/cmip6-cmip6/LR_factor2",
    src="cmip6",
    variables=[
                "air_temperature",
                "u_component_of_wind",
                "v_component_of_wind",
                "pressure_sea_level", # "surface_pressure" for 3H | "pressure_sea_level" for "D"
                "precipitation",
                "specific_humidity",
                "cloud_cover",
                "upward_heat_flux",
                "moisture_in_soil"
               ],
    start_train_year=1960,
    start_val_year=2011,
    start_test_year=2013,
    end_year=2015,
    num_shards=5,  # set 5 for "D" and 20 for "3H"
    frequency="D", # H | 3H | D
)

100%|██████████| 2/2 [00:36<00:00, 18.23s/it]


### EOBS_EOBS task

In [ ]:
# Create LR images with simple bilinear upscaling of HR onew
root_dir="/app/data/raw/e-obs/ensemble_mean/010_grid/"
target_dir = "/app/data/raw/e-obs/ensemble_mean/020_grid/"
SCALE = 2

coarsen_data(root_dir, target_dir, SCALE)

1950-2023 rr.nc
1950-2023 fg.nc
1950-2023 tg.nc
1950-2023 hu.nc
1950-2023 pp.nc
1950-2023 tn.nc
Got regrid exception
1950-2023 qq.nc
Got regrid exception
1950-2023 tx.nc


In [ ]:
# HR data for EOBS-EOBS task
convert_nc2npz(
    root_dir="/app/data/raw/e-obs/ensemble_mean/010_grid/1950-2023",
    save_dir="/app/data/processed/eobs-eobs/010_grid/",
    src="eobs",
    variables=[
            "mean_temperature",
            "maximum_temperature",
            "minimum_temperature",
            "precipitation_sum"
               ],
    start_train_year=1960,
    start_val_year=2018,
    start_test_year=2020,
    end_year=2022,
    num_shards=5,
    frequency="D",
    align_target = None,
    periodic=False # whether data cover all the globe
)

In [ ]:
# LR data for EOBS-EOBS task
convert_nc2npz(
    root_dir="/app/data/raw/e-obs/ensemble_mean/020_grid/1950-2023",
    save_dir="/app/data/processed/eobs-eobs/020_grid/",
    src="eobs",
    variables=[
            "mean_temperature",
            "maximum_temperature",
            "minimum_temperature",
            "precipitation_sum",
            "wind_speed_mean",
            "sea_level_pressure_avg",
            "relative_humidity_avg",
            "global_radiation_mean"
               ],
    start_train_year=1960,
    start_val_year=2018,
    start_test_year=2020,
    end_year=2022,
    num_shards=5,
    frequency="D",
    align_target = None,
    periodic=False # whether data cover all the globe
)

### ERA-EOBS task

In [ ]:
# Create LR images with simple bilinear upscaling
root_dir="/app/data/raw/era5-eobs/era5_0.25_D"
target_dir = "/app/data/raw/era5-eobs/era5_1.00_D"
SCALE = 4
coarsen_data(root_dir, target_dir, SCALE)

In [ ]:
# EOBS data for ERA-EOBS task
convert_nc2npz(
    root_dir="/app/data/raw/era5-eobs/e-obs/ensemble_mean/010_grid/1950-2023/",
    save_dir="/app/data/processed/era5-eobs/e-obs/ensemble_mean/0125_grid/",
    src="eobs",
    variables=[
                "mean_temperature",
                "minimum_temperature",
                "maximum_temperature",
                "precipitation_sum",
                "sea_level_pressure_avg",
                "relative_humidity_avg",
                "global_radiation_mean",
               ],
    start_train_year=1960,
    start_val_year=2018,
    start_test_year=2020,
    end_year=2022,
    num_shards=5, # set 5 for "D" and 20 for "3H"
    frequency="D",
    # align_target = "/app/data/raw/era5-eobs/era5_0.25_D",
    # scale_factor=0.5,
    periodic=False # whether data cover all the globe
)

100%|██████████| 2/2 [01:42<00:00, 51.28s/it]


In [ ]:
# Create mask for E-OBS dataset to exclude NaNs
src_dir='/app/data/processed/era5-eobs/e-obs/ensemble_mean/0125_grid/'

for folder in ["train", "test", "val"]:
    inp_file_list = sorted(
            glob.glob(os.path.join(src_dir, folder, "*.npz"))
        )
    inp_file_list = [f for f in inp_file_list if "climatology" not in f]
    n_files = len(inp_file_list)
    for idx in range(n_files):
        inp = np.load(inp_file_list[idx])
        for k in ["tg", "tx", "tn", "rr"]:
            mask = ~np.isnan(inp[k]).any(axis=0)*1
            try:
                mask_global = (mask_global == 1) & (mask == 1)
            except NameError:
                mask_global = mask == 1
            # print(np.sum(mask_global))

# Save mask
np.save(os.path.join(src_dir, "mask.npy"), mask_global)

In [6]:
# ERA data for ERA-EOBS task
convert_nc2npz(
    root_dir="/app/data/raw/era5-eobs/era5_1.00_D",
    save_dir="/app/data/processed/era5-eobs/era5_1.00_D",
    src="era5",
    variables=[
                "2m_temperature",
                "minimum_temperature",
                "maximum_temperature",
                "rainfall",
               ],
    start_train_year=1960,
    start_val_year=2018,
    start_test_year=2020,
    end_year=2022,
    num_shards=5, # set 5 for "D" and 20 for "3H"
    frequency="D",
    periodic=False # whether data cover all the globe
)

100%|██████████| 2/2 [00:05<00:00,  2.58s/it]


## Model training


Ensure that you have completed the preprocessing of the raw data, as this step requires the preprocessed version.

This stage refers to the configuration files from the `configs/train` folder.

Single run performs the training of the single model. To train another architecture/parameters, edit the configuration file and run training again.

In [ ]:
%run /app/src/benchmark/cmip6_cmip6_dl.py
# %run /app/src/benchmark/era5_era5_dl.py
# %run /app/src/benchmark/era5_eobs_dl.py

## Evaluation

Once you have trained the model you could refer to the checkpoint (will be saved in `base_dir` defined in train config) and test that model with test data. Please, specify the details in `.yaml` file at `configs/inference` folder. As a part of the pipeline you could collect the desired metrics of desired model (list everything the config file) with `save_metrics.py` script. Also, one might plot output of the model with `plots.py` script. The bounds of area of interest could be defined in `configs/inference` folder. Run procedure like this:

In [ ]:
%run /app/src/benchmark/save_metris.py

Pay attention to the necessity to define config path in that script explicitly.

As a result, desired mentrics will be saved as `metrics_avg.pkl` in `base_dir`.

###  Baseline

In [1]:
import sys
sys.path.append("/app/src/benchmark")

from era5_eobs_baseline import load_model_baseline

In [2]:
dm, trainer, (bilinear, bicubic) = load_model_baseline()

# Perform validation and testing for each model
for model, model_name in zip(
    [bilinear],
    ["bilinear-interpolation"],
):
    print("Validating model:", model_name)
    # trainer.validate(model, dataloaders=dm)

    print("Testing model:", model_name)
    trainer.test(model, dataloaders=dm)

Loading architecture: bilinear-interpolation
Using optimizer associated with architecture
Using learning rate scheduler associated with architecture
Loading training loss: mse
Using custom training transform
Loading validation loss: rmse
Loading validation loss: pearson
Loading validation loss: mean_bias
Loading validation loss: mse
Loading validation loss: PSNR
Loading validation loss: SSIM
Loading validation loss: KGE
Using custom validation transform
Using custom validation transform
Using custom validation transform
Using custom validation transform
Using custom validation transform
Using custom validation transform
Using custom validation transform
Loading test loss: rmse
Loading test loss: lat_rmse
Loading test loss: pearson
Loading test loss: lat_mean_bias
Loading test loss: mean_bias
Loading test loss: lat_PSNR
Loading test loss: PSNR
Loading test loss: SSIM
Loading test loss: KGE
Using custom test transform
Using custom test transform
Using custom test transform
Using custom t

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Validating model: bilinear-interpolation
Testing model: bilinear-interpolation


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5]


Output()

/opt/conda/envs/bias_correction/lib/python3.12/site-packages/pytorch_msssim/ssim.py:48: UserWarning: Plan failed 
with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: 
CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  out = conv(out, weight=win.transpose(2 + i, -1), stride=1, padding=0, groups=C)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃         Test metric          ┃         DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      test/KGE:aggregate      │     -0.5068634152412415      │
│         test/KGE:rr          │     -3.9520695209503174      │
│         test/KGE:tg          │      0.7112846374511719      │
│         test/KGE:tn          │      0.401448130607605       │
│         test/KGE:tx          │      0.8118829727172852      │
│     test/PSNR:aggregate      │      32.93587875366211       │
│         test/PSNR:rr         │      39.65045166015625       │
│         test/PSNR:tg         │      31.727224349975586      │
│         test/PSNR:tn         │      28.98265838623047       │
│         test/PSNR:tx         │      31.383180618286133      │
│     test/SSIM:aggregate      │      0.9408464780304614      │
│         test/SSIM:rr         │      0.9342679730935092      │
│         test/SSIM:tg         │      0.947395810139597       │
│         test/SSIM:tn         │      0.9354605045729295      │
│         test/SSIM:tx         │      0.9462616243158105      │
│   test/lat_PSNR:aggregate    │      32.89619408641981       │
│       test/lat_PSNR:rr       │      39.54264279563459       │
│       test/lat_PSNR:tg       │      31.75154994260724       │
│       test/lat_PSNR:tn       │       29.1580556094602       │
│       test/lat_PSNR:tx       │      31.132527997977203      │
│ test/lat_mean_bias:aggregate │      0.3094969170625388      │
│    test/lat_mean_bias:rr     │   -2.2017893883479808e-05    │
│    test/lat_mean_bias:tg     │      0.4539495883824277      │
│    test/lat_mean_bias:tn     │      1.0146162414814779      │
│    test/lat_mean_bias:tx     │     -0.23055614371986694     │
│   test/lat_rmse:aggregate    │      1.3336412818692778      │
│       test/lat_rmse:rr       │    0.0020367329988125746     │
│       test/lat_rmse:tg       │      1.4556257891567608      │
│       test/lat_rmse:tn       │      2.1161211536000137      │
│       test/lat_rmse:tx       │      1.7607814517215243      │
│   test/mean_bias:aggregate   │     0.35566839575767517      │
│      test/mean_bias:rr       │    -2.717570896493271e-05    │
│      test/mean_bias:tg       │      0.506580114364624       │
│      test/mean_bias:tn       │      1.0603444576263428      │
│      test/mean_bias:tx       │     -0.14422382414340973     │
│    test/pearson:aggregate    │      0.902480959892273       │
│       test/pearson:rr        │      0.7186861634254456      │
│       test/pearson:tg        │      0.9757276773452759      │
│       test/pearson:tn        │      0.9393540620803833      │
│       test/pearson:tx        │      0.9761562347412109      │
│     test/rmse:aggregate      │      1.3327443599700928      │
│         test/rmse:rr         │    0.0020131398923695087     │
│         test/rmse:tg         │      1.4639925956726074      │
│         test/rmse:tn         │      2.1588025093078613      │
│         test/rmse:tx         │       1.7061687707901        │
└──────────────────────────────┴──────────────────────────────┘